# Flight Network Analysis and Delay Prediction
Apache Spark + GraphFrames (GraphX API) + Spark MLlib. Run top to bottom; the last cell exports `model_export.json` for the Streamlit app.

In [ ]:
# pip install pyspark==3.5.1 graphframes==0.6 matplotlib pandas numpy

In [ ]:
# STEP 1: Imports, configuration and Spark session
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from graphframes import GraphFrame

DATA_DIR = "datasets"                 # folder with flights.csv, airports.csv, airlines.csv
SAMPLE_FRACTION = 0.2                 # 20% sample (set 1.0 for the full data)
SEED = 42

spark = (SparkSession.builder
         .appName("FlightNetworkDelayPrediction")
         .master("local[*]")
         .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.13")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())
spark.sparkContext.setCheckpointDir("/tmp/graphframes_checkpoint")
spark.sparkContext.setLogLevel("WARN")
print("Spark version :", spark.version)
print("Default parallelism (cores):", spark.sparkContext.defaultParallelism)

In [ ]:
# STEP 2: Load the dataset
flights_raw = spark.read.csv(f"{DATA_DIR}/flights.csv", header=True, inferSchema=True)
airports = spark.read.csv(f"{DATA_DIR}/airports.csv", header=True, inferSchema=True)
airlines = spark.read.csv(f"{DATA_DIR}/airlines.csv", header=True, inferSchema=True)

if SAMPLE_FRACTION < 1.0:
    flights_raw = flights_raw.sample(False, SAMPLE_FRACTION, SEED)

print("Raw flight records :", flights_raw.count())
print("Airports           :", airports.count())
print("Airlines           :", airlines.count())
flights_raw.printSchema()

In [ ]:
# STEP 3: Data cleaning and label creation
# Keep only completed flights between valid 3-letter IATA airports.
# Label: LABEL = 1 if the flight arrived more than 15 minutes late.
iata = r"^[A-Z]{3}$"
flights = (flights_raw
    .withColumn("ORIGIN_AIRPORT", F.col("ORIGIN_AIRPORT").cast("string"))
    .withColumn("DESTINATION_AIRPORT", F.col("DESTINATION_AIRPORT").cast("string"))
    .filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
    .filter(F.col("ARRIVAL_DELAY").isNotNull() & F.col("DISTANCE").isNotNull())
    .filter(F.col("ORIGIN_AIRPORT").rlike(iata) & F.col("DESTINATION_AIRPORT").rlike(iata))
    .withColumn("DEP_HOUR", (F.col("SCHEDULED_DEPARTURE") / 100).cast("int"))
    .withColumn("LABEL", (F.col("ARRIVAL_DELAY") > 15).cast("int"))
    .select("MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
            "DEP_HOUR", "DISTANCE", "ARRIVAL_DELAY", "LABEL")
    .cache())

print("Flights after cleaning:", flights.count())
flights.groupBy("LABEL").count().show()

In [ ]:
# STEP 4: Exploratory analysis with Spark SQL / DataFrame API
flights.agg(F.count("*").alias("flights"),
            F.round(F.avg("LABEL") * 100, 2).alias("delayed_pct"),
            F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay_min")).show()

airline_names = airlines.withColumnRenamed("AIRLINE", "AIRLINE_NAME")
by_airline = (flights.groupBy("AIRLINE")
    .agg(F.count("*").alias("flights"),
         F.round(F.avg("LABEL") * 100, 2).alias("delayed_pct"),
         F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay"))
    .join(airline_names, F.col("AIRLINE") == F.col("IATA_CODE"))
    .select("AIRLINE", "AIRLINE_NAME", "flights", "delayed_pct", "avg_arrival_delay")
    .orderBy(F.desc("delayed_pct")))
by_airline.show(truncate=False)

by_month = (flights.groupBy("MONTH").agg(F.round(F.avg("LABEL") * 100, 2).alias("delayed_pct"))
            .orderBy("MONTH").toPandas())
by_hour = (flights.groupBy("DEP_HOUR").agg(F.round(F.avg("LABEL") * 100, 2).alias("delayed_pct"))
           .orderBy("DEP_HOUR").toPandas())
by_airline_pd = by_airline.toPandas()

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].bar(by_airline_pd["AIRLINE"], by_airline_pd["delayed_pct"], color="#2b6cb0")
ax[0].set_title("Delayed flights (%) by airline")
ax[1].plot(by_month["MONTH"], by_month["delayed_pct"], marker="o", color="#c05621")
ax[1].set_title("Delayed flights (%) by month")
ax[1].set_xticks(range(1, 13))
ax[2].plot(by_hour["DEP_HOUR"], by_hour["delayed_pct"], marker="o", color="#2f855a")
ax[2].set_title("Delayed flights (%) by scheduled departure hour")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("eda_delay_patterns.png", dpi=150)
plt.show()

In [ ]:
# STEP 5: Build the airport network (vertices = airports, edges = routes)
vertices = airports.select(F.col("IATA_CODE").alias("id"), "AIRPORT", "CITY", "STATE")

edges = (flights.groupBy(F.col("ORIGIN_AIRPORT").alias("src"),
                         F.col("DESTINATION_AIRPORT").alias("dst"))
         .agg(F.count("*").alias("num_flights"),
              F.round(F.avg("DISTANCE"), 1).alias("distance")))

# keep only edges whose endpoints exist in the airport table
ids = vertices.select("id")
edges = (edges.join(ids.withColumnRenamed("id", "src"), "src")
              .join(ids.withColumnRenamed("id", "dst"), "dst"))
used = edges.select("src").union(edges.select("dst")).distinct().withColumnRenamed("src", "id")
vertices = vertices.join(used, "id")

g = GraphFrame(vertices, edges)
print("Airports (vertices):", g.vertices.count())
print("Routes (edges)     :", g.edges.count())

In [ ]:
# STEP 6: Graph analytics (PageRank, degrees, components, shortest paths)
in_deg = g.inDegrees
out_deg = g.outDegrees

pr = g.pageRank(resetProbability=0.15, maxIter=10)
pagerank = pr.vertices.select("id", "AIRPORT", "CITY", "pagerank")

hubs = (pagerank.join(in_deg, "id", "left").join(out_deg, "id", "left")
        .fillna(0, subset=["inDegree", "outDegree"]).cache())

print("Top 10 airports by PageRank (most critical hubs):")
hubs.orderBy(F.desc("pagerank")).show(10, truncate=False)

print("Top 10 airports by number of outgoing routes:")
hubs.orderBy(F.desc("outDegree")).select("id", "CITY", "inDegree", "outDegree").show(10)

print("Top 10 busiest routes:")
g.edges.orderBy(F.desc("num_flights")).show(10)

print("Connected components (network connectivity):")
cc = g.connectedComponents()
cc.groupBy("component").count().orderBy(F.desc("count")).show()

print("Shortest path (number of hops) to landmark hubs ATL, LAX, JFK:")
g.shortestPaths(landmarks=["ATL", "LAX", "JFK"]).select("id", "distances").show(10, truncate=False)

top_pr = hubs.orderBy(F.desc("pagerank")).limit(15).toPandas()
plt.figure(figsize=(9, 4.5))
plt.bar(top_pr["id"], top_pr["pagerank"], color="#6b46c1")
plt.title("Top 15 airports by PageRank")
plt.ylabel("PageRank score")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("graph_top_pagerank.png", dpi=150)
plt.show()

In [ ]:
# STEP 7: Feature engineering - join graph metrics back to each flight
metrics = hubs.select("id", "pagerank", "inDegree", "outDegree")

origin_m = metrics.select(F.col("id").alias("ORIGIN_AIRPORT"),
                          F.col("pagerank").alias("ORIGIN_PR"),
                          F.col("inDegree").alias("ORIGIN_IN"),
                          F.col("outDegree").alias("ORIGIN_OUT"))
dest_m = metrics.select(F.col("id").alias("DESTINATION_AIRPORT"),
                        F.col("pagerank").alias("DEST_PR"),
                        F.col("inDegree").alias("DEST_IN"),
                        F.col("outDegree").alias("DEST_OUT"))

data = (flights.join(F.broadcast(origin_m), "ORIGIN_AIRPORT")
               .join(F.broadcast(dest_m), "DESTINATION_AIRPORT")
               .cache())
print("Rows in modelling dataset:", data.count())

In [ ]:
# STEP 8: Train / test split and class-balancing weights
# Only information known BEFORE departure is used as features (no delay columns).
train, test = data.randomSplit([0.8, 0.2], seed=SEED)
pos_ratio = train.filter("LABEL = 1").count() / train.count()
train = train.withColumn("WEIGHT",
                         F.when(F.col("LABEL") == 1, 1.0 - pos_ratio).otherwise(pos_ratio))
print(f"Delayed share in training data: {pos_ratio:.3f}")

num_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE",
            "ORIGIN_PR", "ORIGIN_IN", "ORIGIN_OUT", "DEST_PR", "DEST_IN", "DEST_OUT"]

indexer = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_IDX", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["AIRLINE_IDX"], outputCols=["AIRLINE_VEC"])
assembler = VectorAssembler(inputCols=["AIRLINE_VEC"] + num_cols,
                            outputCol="raw_features", handleInvalid="skip")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="LABEL", weightCol="WEIGHT",
                        maxIter=30, regParam=0.01)
rf = RandomForestClassifier(featuresCol="raw_features", labelCol="LABEL", weightCol="WEIGHT",
                            numTrees=50, maxDepth=7, seed=SEED)

lr_pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])
rf_pipeline = Pipeline(stages=[indexer, encoder, assembler, rf])

In [ ]:
# STEP 9: Train and evaluate both models
def evaluate(name, pred, train_time):
    auc = BinaryClassificationEvaluator(labelCol="LABEL", rawPredictionCol="rawPrediction",
                                        metricName="areaUnderROC").evaluate(pred)
    mc = MulticlassClassificationEvaluator(labelCol="LABEL", predictionCol="prediction")
    return {"Model": name,
            "AUC": round(auc, 4),
            "Accuracy": round(mc.setMetricName("accuracy").evaluate(pred), 4),
            "F1": round(mc.setMetricName("f1").evaluate(pred), 4),
            "Precision": round(mc.setMetricName("weightedPrecision").evaluate(pred), 4),
            "Recall": round(mc.setMetricName("weightedRecall").evaluate(pred), 4),
            "Train time (s)": round(train_time, 1)}

models, preds, rows = {}, {}, []
for name, pipe in [("Logistic Regression", lr_pipeline), ("Random Forest", rf_pipeline)]:
    t0 = time.time()
    models[name] = pipe.fit(train)
    elapsed = time.time() - t0
    preds[name] = models[name].transform(test).cache()
    rows.append(evaluate(name, preds[name], elapsed))
    print(f"{name} trained in {elapsed:.1f}s")

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("model_results.csv", index=False)

In [ ]:
# STEP 10: Confusion matrix and feature importance
best = results.sort_values("AUC", ascending=False).iloc[0]["Model"]
print("Best model by AUC:", best)

cm = (preds[best].groupBy("LABEL", "prediction").count().toPandas()
      .pivot(index="LABEL", columns="prediction", values="count").fillna(0))
print(cm)

plt.figure(figsize=(4.5, 4))
plt.imshow(cm.values, cmap="Blues")
plt.xticks(range(cm.shape[1]), [f"Pred {int(c)}" for c in cm.columns])
plt.yticks(range(cm.shape[0]), [f"Actual {int(i)}" for i in cm.index])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, f"{int(cm.values[i, j]):,}", ha="center", va="center")
plt.title(f"Confusion matrix - {best}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# Random Forest feature importance with readable names
rf_model = models["Random Forest"].stages[-1]
scored = models["Random Forest"].transform(test.limit(10))
attrs = scored.schema["raw_features"].metadata["ml_attr"]["attrs"]
names = {a["idx"]: a["name"] for group in attrs.values() for a in group}
imp = pd.Series(rf_model.featureImportances.toArray(),
                index=[names[i] for i in range(len(names))]).sort_values(ascending=False)
print(imp.head(10))

imp.head(10)[::-1].plot(kind="barh", figsize=(8, 4.5), color="#2f855a")
plt.title("Random Forest - top 10 feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

In [ ]:
# STEP 11: Wrap up
print("Project finished. Saved files: eda_delay_patterns.png, graph_top_pagerank.png,")
print("model_results.csv, confusion_matrix.png, feature_importance.png")
# spark.stop()   # run only when completely done

In [ ]:
# STEP 12: Export the trained models for the Streamlit app (creates model_export.json)
import json
import numpy as np

rf_pm, lr_pm = models["Random Forest"], models["Logistic Regression"]
rf_model, lr_model = rf_pm.stages[-1], lr_pm.stages[-1]
airline_labels = list(rf_pm.stages[0].labelsArray[0])

sample = rf_pm.transform(test.limit(5))
attrs = sample.schema["raw_features"].metadata["ml_attr"]["attrs"]
names = {a["idx"]: a["name"] for grp in attrs.values() for a in grp}
feature_names = [names[i] for i in range(len(names))]

def walk(node):
    if node.getClass().getSimpleName() == "InternalNode":
        sp = node.split()
        if sp.getClass().getSimpleName() == "ContinuousSplit":
            return [int(sp.featureIndex()), 0, round(float(sp.threshold()), 6),
                    walk(node.leftChild()), walk(node.rightChild())]
        return [int(sp.featureIndex()), 1, [float(c) for c in sp.leftCategories()],
                walk(node.leftChild()), walk(node.rightChild())]
    st = [float(s) for s in node.impurityStats().stats()]
    return round(st[1] / sum(st), 4) if sum(st) else 0.0

rf_trees = [walk(t._java_obj.rootNode()) for t in rf_model.trees]

def run_tree(t, x):
    while isinstance(t, list):
        f, k, v, l, r = t
        t = l if ((x[f] <= v) if k == 0 else (x[f] in v)) else r
    return t
row = rf_pm.transform(test.limit(1)).select("raw_features", "probability").first()
x = row["raw_features"].toArray()
print("exported:", sum(run_tree(t, x) for t in rf_trees) / len(rf_trees),
      "| spark:", row["probability"][1])

lr_export = {"coef": lr_model.coefficients.toArray().tolist(),
             "intercept": float(lr_model.intercept),
             "std": lr_pm.stages[3].std.toArray().tolist()}

ap = airports.select("IATA_CODE", "AIRPORT", "CITY", "STATE", "LATITUDE", "LONGITUDE").toPandas()
hm = hubs.select("id", "pagerank", "inDegree", "outDegree").toPandas()
meta = ap.merge(hm, left_on="IATA_CODE", right_on="id")
nn = lambda v: None if pd.isna(v) else float(v)
airport_list = [dict(code=r.IATA_CODE, name=r.AIRPORT, city=r.CITY, state=r.STATE,
                     lat=nn(r.LATITUDE), lon=nn(r.LONGITUDE), pr=round(float(r.pagerank), 4),
                     inn=int(r.inDegree), out=int(r.outDegree)) for r in meta.itertuples()]
routes = {f"{r.src}-{r.dst}": float(r.distance)
          for r in g.edges.select("src", "dst", "distance").toPandas().itertuples()}

export = {
    "airline_labels": airline_labels, "feature_names": feature_names,
    "rf": rf_trees, "lr": lr_export, "airports": airport_list, "routes": routes,
    "airline_stats": by_airline_pd.to_dict("records"),
    "by_month": by_month.to_dict("records"), "by_hour": by_hour.to_dict("records"),
    "results": results.to_dict("records"),
    "cm": {"tn": int(cm.loc[0, 0.0]), "fp": int(cm.loc[0, 1.0]),
           "fn": int(cm.loc[1, 0.0]), "tp": int(cm.loc[1, 1.0])},
    "importance": imp.head(10).round(4).to_dict(),
    "totals": {"flights": int(flights.count()), "delayed_pct": 18.46,
               "airports": int(g.vertices.count()), "routes": int(g.edges.count())},
}
def conv(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))
with open("model_export.json", "w") as f:
    json.dump(export, f, separators=(",", ":"), default=conv)
print("model_export.json:", round(os.path.getsize("model_export.json") / 1024), "KB")